Author: Thomas Joyce

Class: ASTR 400B - Galaxies and Cosmology

Description: Workbook documenting Homework 6

--------------------------------------------------------------------------------------------------------------------------------

In [22]:
### Importations ###
# Numpy and Astropy
import numpy as np
# Plotting
import matplotlib.pyplot as plt
# External (must be in same directory as Homework6)
from CenterOfMass import CenterOfMass

In [23]:
def OrbitCOM(galaxy, start, end, n):
        ''' 
        Description: Designs a txt file in a 7 column format, 
        tracking the time, position, and velocity of a galaxies center of mass
            
        Inputs:
            - galaxy: string name of galxy (ex: "MW")
            - start: the interger number corresponding to the first snapshot to be read in
            - end: the interget number correponding to the last snapshot to be read in
            - n: an interger number corresponding to the amout of intervals over which the COM will be returned
            
        '''
        # Defining relevant COM parameters #
        Delta = 0.1
        VolDec = 2
        if galaxy == 'M33':
            VolDec = 4
        
        
        # Defining Output Filename #
        OutputName = 'Orbit_' + galaxy + '.txt'
        
        snap_ids = np.array(np.arange(start, end, n)) # array of snap designations to be read in
        
        # Check to end function if equal divisions of n are not achieved #
        if abs(end - start) % n != 0:
            print('Interval is incompatible!!!')
            print('Function Ending ...')
            return
        
        orbit = np.zeros([len(snap_ids), 7]) # defining output list to store values
        
        # For loop to iterate through snap shots #
        for i, snap_id in enumerate(snap_ids):
            # Determining File Name #
            ilbl = '000' + str(snap_id) # adding snap number to the value '000'
            ilbl = ilbl[-3:] # only the last 3 digits being used
            
            Filename = '%s_'%(galaxy) + ilbl + '.txt' # reconstructing filename
            
            # Creating Center of Mass Object #
            gal_COM = CenterOfMass(FileName, 2) # Center of mass object using disk particles
            
            # Computing center of mass for object #
            gal_P = gal_COM.COM_P(Delta, VolDec)
            gal_V = gal_COM.COM_V(gal_P[0], gal_P[1], gal_P[2])
            
            Time = gal_COM.time / 1000 # Time in Gyr of Snapshot
            
            # Storing values to orbit array #
            orbit[i][0] = Time.value # time is the first index (column)
            
            orbit[i][1] = gal_P[0].value # x position is the second index (column)
            orbit[i][2] = gal_P[1].value # y position is the third index (column)
            orbit[i][3] = gal_P[2].value # z position is the fourth index (column)
            
            orbit[i][4] = gal_V[0].value # x velocity is the fifth index (column)
            orbit[i][5] = gal_V[1].value # y velocity is the sixth index (column)
            orbit[i][6] = gal_V[2].value # z velocity is the seventh index (column)
            
            # Print statmeent for code running visibility #
            print('Iteration: ' + str(i) + ' / ' + str(len(snap_ids) - 1))
            
        # Saving to txt file #
        np.savetxt(OutputName, orbit, fmt = "%11.3f"*7, comments = '#', 
                   header = "{:10s}{:11s}{:11s}{:11s}{:11s}{:11s}{:11s}".format('t', 'x', 'y', 'z', 'vx', 'vy', 'vz'))
### END OrbitCOM
    
            

In [24]:
# Generating Orbit txt Files #

# Milky Way Orbits #
#OrbitCOM('MW', 0, 800, 5)

# M31 Orbits #
#OrbitCOM('M31', 0, 800, 5)

# M33 Orbits #
#OrbitCOM('M33', 0, 800, 5)

### NOTE ###
''' Don't bother running these lines, I left all the data txt file sout of 
the repository to prevent a mass upload to git hub, I do have the resulting Orbit
txt files with each iteration saved in the directory '''

" Don't bother running these lines, I left all the data txt file sout of \nthe repository to prevent a mass upload to git hub, I do have the resulting Orbit\ntxt files with each iteration saved in the directory "

In [25]:
def ReadOrbitFile(FileName):
    ''' 
    Description: reads orbit txt file generated from OrbitCOM and saves the data
    into variables Time, Gal_pos, and Gal_vec 
    
    Inputs:
        - FileName: File name to be read in (string)
        
    Returns:
        - Time: a list of time values in Gyr (NON Astropy Quantity)
        - Gal_Pos: an array of COM position vectors for the galaxy in kpc, the data
        is formatted as Gal_Pos = [ [COM X positions], [COM Y positions], [COM Z positions] ]
        (NON Astropy Quantity)
        - Gal_Vec: an array of COM velocity vectors for the galaxy in km/s, the data
        is formatted as Gal_Vec = [ [COM X velocity], [COM Y velocity], [COM Z velocity] ]
        (NON Astropy Quantity)'''
    data = np.genfromtxt(FileName, dtype = None, names = True, skip_header = 0) # data array creation
    
    Time = data['t']
    Gal_Pos = [ data['x'], data['y'], data['z'] ]
    Gal_Vec = [ data['vx'], data['vy'], data['vz'] ]
    
    return Time, Gal_pos, Gal_Vec
### END ReadOrbitFile

In [26]:
def RelativeSeperation(Gal1, Gal2):
    ''' 
    Description: Determines the relative seperation of a state vector for Galaxy 1 (Gal1)
    and Galaxy 2 (Gal2). This can be done with position (giving the relative position seperation) and
    with velocity (giving ht relative velocity between the two)
    
    Inputs:
        - Gal1: State vector for galaxy 1, list with indeces [x_component, y_component, z_component]
        - Gal2: State vector for galaxy 2, list with indeces [x_component, y_component, z_component]
        
    Returns:
        - SeperationMagnitude: Magnitude of the difference between the two state vectors
    '''
    
    # Computing X Seperation #
    X_sep = Gal2[0] - Gal1[0]
    
    # Computing Y Seperation #
    Y_sep = Gal2[1] - Gal1[1]
    
    # Computing Z Seperation #
    Z_sep = Gal2[2] - Gal1[2]
    
    SeperationMagnitude = np.sqrt((X_sep ** 2) + (Y_sep ** 2) + (Z_sep ** 2))
    
    return SeperationMagnitude
### END RelativeSeperation

In [18]:
def PlotRelativeSeperation(Gal1_FileName, Gal2_FileName, mode):
    ''' '''
    
    G1_time, G1_pos, G1_vec = ReadOrbitFile(Gal1_FileName)
    G2_time, G2_pos, G2_vec = ReadOrbitFile(Gal2_FileName)
    
    # we have to see how the data is formatted but lets say we packaged 
    # into position and velocity statvecots as elements of a list
    
    ### basically we have Time = list of time values in Gyr
    ### Gal1_pos = [P1, P2, P3 ...] where Pi = [Pos_x, Pos_y, Pos_z]
    ### --> same for gal 2
    
    ### rememeber we have to iterate through these lists to find the relative seperation at each index
    
    Gal1_name = Gal1_FileName[6:-4]
    Gal2_name = Gal2_FileName[6:-4]
    
    # Figure #
    plt.figure('Relative Seperation Plot', figsize = (6,6), dpi = 250)
    
    # Plotting (Particle 1)#
    plt.scatter(Time, SepMag, marker = '.', s = 80, color = 'black') #,zorder = 2, label = 'Dark Matter') 

    
    # Plot Formatting #
    if mode == 'Distance':
        Ylabel = 'Relative Seperation [kpc]'
        Title = 'Relative Distance Between ' + Gal1_name + ' and ' + Gal2_name
        Output = Gal1_name + '_' + Gal2_name + 'RelativeDistance.png'
    if mode == 'Velocity':
        Ylabel = 'Relative Velocity [km/s]'
        Title = 'Relative Velocity Between ' + Gal1_name + ' and ' + Gal2_name
        Output = Gal1_name + '_' + Gal2_name + 'RelativeVelocity.png'
    
    plt.xlabel('Time [Gyr]', fontsize = 15)
    plt.ylabel(Ylabel, fontsize = 15)
    plt.title(Title)
    
    plt.grid(axis = 'both', color = 'grey', linewidth = 0.7, linestyle=':')
    
    #plt.legend()
    
    plt.tight_layout()
    
    # Plot Saving #
    plt.savefig(Output)
### END PlotRelativeSeperatio

# Part 3: Plotting #

Plotting the graphs requested for part 3

In [19]:
### code to run the plotting and stuff
### 'Orbit_' + galaxy + '.txt'

# Relative Seperation of MW and M31 #
PlotRelativeSeperation('Orbit_MW.txt', 'Orbit_M31.txt', 'Distance')

[-2.0800e+00 -1.8200e+00 -1.9000e+00 -2.1600e+00 -2.4500e+00 -2.7900e+00
 -3.3300e+00 -3.5700e+00 -3.6700e+00 -3.9500e+00 -4.0900e+00 -4.4600e+00
 -5.1300e+00 -5.9200e+00 -6.5600e+00 -6.9400e+00 -7.4500e+00 -7.6800e+00
 -8.6700e+00 -9.4000e+00 -9.9900e+00 -1.0780e+01 -1.1570e+01 -1.2410e+01
 -1.3330e+01 -1.4150e+01 -1.5250e+01 -1.6690e+01 -1.8230e+01 -1.9840e+01
 -2.1250e+01 -2.2640e+01 -2.4220e+01 -2.5930e+01 -2.7620e+01 -2.9420e+01
 -3.1120e+01 -3.3120e+01 -3.5000e+01 -3.6910e+01 -3.9100e+01 -4.1230e+01
 -4.3060e+01 -4.5410e+01 -4.7660e+01 -4.9650e+01 -5.2290e+01 -5.5290e+01
 -5.8470e+01 -6.1670e+01 -6.4650e+01 -6.7790e+01 -7.0860e+01 -7.4050e+01
 -7.6970e+01 -7.8630e+01 -7.6710e+01 -7.1240e+01 -6.4430e+01 -5.8650e+01
 -5.3660e+01 -4.9110e+01 -4.4710e+01 -4.0550e+01 -3.6460e+01 -3.2660e+01
 -2.9010e+01 -2.5430e+01 -2.1880e+01 -1.8520e+01 -1.5450e+01 -1.2620e+01
 -9.9300e+00 -7.4500e+00 -5.1400e+00 -2.6700e+00 -1.2000e-01  2.3700e+00
  4.7200e+00  6.7200e+00  8.0100e+00  8.1300e+00  4

NameError: name 'plt' is not defined

In [ ]:
# Relative Seperation of M33 and M31 #
PlotRelativeSeperation('Orbit_M33.txt', 'Orbit_M31.txt', 'Distance')

In [ ]:
# Relative Velocity of MW and M31 #
PlotRelativeSeperation('Orbit_MW.txt', 'Orbit_M31.txt', 'Velocity')

In [ ]:
# Relative Velocity of M33 and M31 #
PlotRelativeSeperation('Orbit_M33.txt', 'Orbit_M31.txt', 'Velocity')

# Part 4: Questions #

1)

2)

3)

4)